# Task 1b — Feature Transformation and Linear Regression

Implementation of nonlinear feature transformations followed by linear regression.
The original five input features are expanded to 21 features using linear, quadratic,
exponential, cosine, and constant basis functions. Regularization is evaluated using
10-fold cross-validation.

ETH Zürich — Introduction to Machine Learning, Spring 2024

In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
# Add any additional imports here (however, the task is solvable without using 
# any additional imports)
# import ...

 #### Loading data

In [12]:
data = pd.read_csv("train.csv")
y = data["y"].to_numpy()
data = data.drop(columns=["Id", "y"])
# print a few data samples
print(data.head())
X = data.to_numpy()

     x1    x2    x3    x4    x5
0  0.02  0.05 -0.09 -0.43 -0.08
1 -0.13  0.11 -0.08 -0.29 -0.03
2  0.08  0.06 -0.07 -0.41 -0.03
3  0.02 -0.12  0.01 -0.43 -0.02
4 -0.14 -0.12 -0.08 -0.02 -0.08


#### Transform data

In [13]:
"""
Transform the 5 input features of matrix X (x_i denoting the i-th component of X) 
into 21 new features phi(X) in the following manner:
5 linear features: phi_1(X) = x_1, phi_2(X) = x_2, phi_3(X) = x_3, phi_4(X) = x_4, phi_5(X) = x_5
5 quadratic features: phi_6(X) = x_1^2, phi_7(X) = x_2^2, phi_8(X) = x_3^2, phi_9(X) = x_4^2, phi_10(X) = x_5^2
5 exponential features: phi_11(X) = exp(x_1), phi_12(X) = exp(x_2), phi_13(X) = exp(x_3), phi_14(X) = exp(x_4), phi_15(X) = exp(x_5)
5 cosine features: phi_16(X) = cos(x_1), phi_17(X) = cos(x_2), phi_18(X) = cos(x_3), phi_19(X) = cos(x_4), phi_20(X) = cos(x_5)
1 constant feature: phi_21(X)=1

Parameters
----------
X: matrix of floats, dim = (700,5), inputs with 5 features

Compute
----------
X_transformed: array of floats: dim = (700,21), transformed input with 21 features
"""
X_transformed = np.zeros((700, 21))
# TODO: Enter your code here
for i in range(len(X_transformed)):
    X_transformed[i] = [X[i][0], X[i][1], X[i][2], X[i][3], X[i][4],
                        X[i][0]**2, X[i][1]**2, X[i][2]**2, X[i][3]**2, X[i][4]**2,
                        np.exp(X[i][0]), np.exp(X[i][1]), np.exp(X[i][2]), np.exp(X[i][3]), np.exp(X[i][4]),
                        np.cos(X[i][0]), np.cos(X[i][1]), np.cos(X[i][2]), np.cos(X[i][3]), np.cos(X[i][4]),
                        1]
assert X_transformed.shape == (700, 21)

#### Fit data

In [18]:
"""
Use the transformed data points X_transformed and fit the linear regression on this 
transformed data. Finally, compute the weights of the fitted linear regression. 

Parameters
----------
X_transformed: array of floats: dim = (700,21), transformed input with 21 features
y: array of floats, dim = (700,), input labels)

Compute
----------
w: array of floats: dim = (21,), optimal parameters of linear regression
"""

def fit(X, y, lam):
    w = np.zeros((21,)) #Is this one really needed?
    I_d = np.eye(X.shape[1]) # Creating the identity matrix I_d (dimension of attributes)
    XT = np.transpose(X)
    XTX = np.dot(XT,X)
   # I_d = np.eye(X.shape[1]) # Creating the identity matrix I_d (dimension of attributes)
    Inv = np.linalg.inv(XTX+lam*I_d)
    XTy = np.dot(XT,y) 
    w = np.dot(Inv, XTy)#Solving the closed system to obtain the optimal weights
    #print(w)
    assert w.shape == (21,)
    return w


def calculate_RMSE(w, X, y):
    RMSE = 0
    y_hat = np.matmul(X,w)
    
    for i in range(0, len(y)):
        RMSE += (y[i] - y_hat[i]) ** 2
        
    RMSE = np.sqrt(RMSE / len(y))
    
    assert np.isscalar(RMSE), 'RMSE is not a scalar'
    return RMSE


w = np.zeros((21,))

assert w.shape == (21,)


#lambdas = [0.1, 1, 10, 100, 200]
lambdas = [8, 9, 10, 11, 12] #lambda = 11
n_folds = 10

RMSE_mat = np.zeros((n_folds, len(lambdas)))

k_folds = KFold(n_splits = n_folds, shuffle = True, random_state = 42)

# Iterate through each fold
for fold, (train_index, test_index) in enumerate(k_folds.split(X_transformed)):
    #creating test and validation sets
    X_train, X_test = X_transformed[train_index], X_transformed[test_index] 
    y_train, y_test = y[train_index], y[test_index]
    #iterate through each lambda
    for i in range(len(lambdas)):
        # Train the model on the training data
        w = fit(X_train, y_train, lambdas[i])
        # compute validation error in test set
        val_error = calculate_RMSE(w, X_test, y_test)
        RMSE_mat[fold][i] = val_error

print(RMSE_mat)
print()
avg_RMSE = np.mean(RMSE_mat, axis=0) # avg_RMSE: array of floats: dim = (5,), average RMSE value for every lambda
print(avg_RMSE)



# TODO: Enter your code here
assert w.shape == (21,)

[[1.75562963 1.75495619 1.75436415 1.75383767 1.75336475]
 [1.90891225 1.90952487 1.9100522  1.91050758 1.91090189]
 [2.24235821 2.24236256 2.24237906 2.24240369 2.24243386]
 [1.98641275 1.98626451 1.98614962 1.98605851 1.98598466]
 [1.65548361 1.65481898 1.65425511 1.65377135 1.65335231]
 [1.86110044 1.86012856 1.8593118  1.85861902 1.8580269 ]
 [2.29113896 2.29057639 2.29009725 2.28968614 2.28933113]
 [2.01148264 2.01235359 2.01313451 2.01383939 2.01447953]
 [1.76675347 1.76721111 1.76762069 1.76798623 1.7683119 ]
 [2.00477997 2.0055418  2.00622121 2.00683086 2.00738121]]

[1.94840519 1.94837386 1.94835856 1.94835404 1.94835681]


In [20]:
# Save results in the required format
np.savetxt("./results_new2.csv", w, fmt="%.12f")